# ATP Match Predictor

Predicts the winner of an ATP match from rankings, recent form, surface record, head-to-head history, age and height, using a scikit-learn gradient-boosted classifier (`HistGradientBoostingClassifier`).

| Section | What it does |
|---|---|
| 1. Data | Downloads and cleans Jeff Sackmann's ATP match data |
| 2. Features | Builds leakage-free features (form, surface, head-to-head) |
| 3. Evaluation | Chronological test, baselines, calibration, bootstrap confidence intervals |
| 4. Predictions | Trains the final model and predicts any matchup |
| 5. Sanity checks | Quick tests that the code behaves correctly |
| 6. Website export | Writes `predictions.json` for the website (optional) |

In [1]:
# ---- Settings you can change ----
START_YEAR = 2019      # first season to download
END_YEAR = None        # None = current year (seasons not yet published are skipped)
TRAIN_FRAC = 0.8       # first 80% of matches (by date) train the evaluation model; the last 20% test it
SEED = 42

In [2]:
import datetime as dt
import io
import json

import numpy as np
import pandas as pd
import requests
from collections import namedtuple

from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Data
Match results come from Jeff Sackmann's ATP archive (one CSV per season). Rows missing rank, ranking points, age or surface are dropped; a missing height defaults to 180 cm.

In [3]:
DATA_URL = ("https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/"
            "main/atp/atp_matches_{year}.csv")
DEFAULT_HEIGHT_CM = 180
REQUIRED = ["winner_rank", "loser_rank", "surface", "winner_rank_points",
            "loser_rank_points", "winner_age", "loser_age"]


def load_raw(start_year, end_year=None):
    """All matches from start_year to end_year, sorted by date. Missing seasons are skipped."""
    end_year = end_year or dt.date.today().year
    frames = []
    for year in range(start_year, end_year + 1):
        try:
            resp = requests.get(DATA_URL.format(year=year), timeout=30)
            resp.raise_for_status()
            df = pd.read_csv(io.StringIO(resp.text))
            df["season"] = year
            frames.append(df)
            print(f"  fetched {year}: {len(df)} matches")
        except Exception as e:
            print(f"  skipped {year}: {e}")
    if not frames:
        raise RuntimeError("No seasons could be downloaded.")
    raw = pd.concat(frames, ignore_index=True)
    raw["tourney_date"] = pd.to_datetime(raw["tourney_date"], format="%Y%m%d")
    return raw.sort_values("tourney_date", kind="stable").reset_index(drop=True)


def clean_matches(raw):
    m = raw.dropna(subset=REQUIRED).copy()
    m[["winner_ht", "loser_ht"]] = m[["winner_ht", "loser_ht"]].fillna(DEFAULT_HEIGHT_CM)
    return m.reset_index(drop=True)


def latest_snapshots(matches):
    """Each player's most recent known id/rank/points/age/height, indexed by name."""
    def side(p):
        d = matches[["tourney_date", f"{p}_name", f"{p}_id", f"{p}_rank",
                     f"{p}_rank_points", f"{p}_age", f"{p}_ht"]].copy()
        d.columns = ["date", "name", "id", "rank", "pts", "age", "ht"]
        return d
    both = pd.concat([side("winner"), side("loser")]).sort_values("date", kind="stable")
    return both.drop_duplicates("name", keep="last").set_index("name")


raw = load_raw(START_YEAR, END_YEAR)
matches = clean_matches(raw)
print(f"\nMatches fetched: {len(raw):,} | usable after cleaning: {len(matches):,} "
      f"| latest match: {matches['tourney_date'].max().date()}")

  fetched 2019: 2806 matches


  fetched 2020: 1462 matches
  fetched 2021: 2733 matches


  fetched 2022: 2917 matches


  fetched 2023: 2986 matches


  fetched 2024: 3076 matches


  fetched 2025: 2944 matches
  fetched 2026: 1449 matches



Matches fetched: 20,373 | usable after cleaning: 20,001 | latest match: 2026-05-25


## 2. Features
Matches are processed in date order, and each match's features use only the matches **before** it (the player history is updated after the features are recorded). Each match is randomly assigned to side A or B, so the model can't learn that "column A always wins." Every feature is a difference, A minus B, so positive values favor player A.

In [4]:
FEATURES = ["rank_diff", "rank_points_diff", "age_diff", "height_diff",
            "form_diff", "surface_form_diff", "h2h_a_winrate"]
Player = namedtuple("Player", "id rank pts age ht")


class PlayerState:
    """Rolling win/loss history used for form, surface and head-to-head features."""

    def __init__(self, form_window=10):
        self.form_window = form_window
        self.results = {}   # player_id -> [1/0, ...]
        self.surface = {}   # (player_id, surface) -> [1/0, ...]
        self.h2h = {}       # sorted (id1, id2) -> {player_id: wins}

    def form(self, pid):
        h = self.results.get(pid, [])[-self.form_window:]
        return sum(h) / len(h) if h else 0.5

    def surface_winrate(self, pid, surface):
        h = self.surface.get((pid, surface), [])
        return sum(h) / len(h) if h else 0.5

    def h2h_winrate(self, a, b):
        wins = self.h2h.get(tuple(sorted((a, b))), {})
        total = wins.get(a, 0) + wins.get(b, 0)
        return wins.get(a, 0) / total if total else 0.5

    def update(self, winner_id, loser_id, surface):
        self.results.setdefault(winner_id, []).append(1)
        self.results.setdefault(loser_id, []).append(0)
        self.surface.setdefault((winner_id, surface), []).append(1)
        self.surface.setdefault((loser_id, surface), []).append(0)
        wins = self.h2h.setdefault(tuple(sorted((winner_id, loser_id))), {})
        wins[winner_id] = wins.get(winner_id, 0) + 1


def pair_features(state, a, b, surface):
    """Feature row for 'player a vs player b'. Positive values favor player a."""
    return {
        "rank_diff": b.rank - a.rank,
        "rank_points_diff": a.pts - b.pts,
        "age_diff": a.age - b.age,
        "height_diff": a.ht - b.ht,
        "form_diff": state.form(a.id) - state.form(b.id),
        "surface_form_diff": state.surface_winrate(a.id, surface) - state.surface_winrate(b.id, surface),
        "h2h_a_winrate": state.h2h_winrate(a.id, b.id),
    }


def build_dataset(matches, seed=SEED):
    rng = np.random.default_rng(seed)
    state = PlayerState()
    rows = []
    for m in matches.itertuples(index=False):
        w = Player(m.winner_id, m.winner_rank, m.winner_rank_points, m.winner_age, m.winner_ht)
        l = Player(m.loser_id, m.loser_rank, m.loser_rank_points, m.loser_age, m.loser_ht)
        a, b, label = (w, l, 1) if rng.random() < 0.5 else (l, w, 0)
        row = pair_features(state, a, b, m.surface)
        row["label"] = label
        row["tourney_date"] = m.tourney_date
        rows.append(row)
        state.update(w.id, l.id, m.surface)   # update AFTER recording features
    return pd.DataFrame(rows), state


def make_model():
    return HistGradientBoostingClassifier(
        max_iter=200, max_depth=3, learning_rate=0.05,
        early_stopping=True, random_state=SEED)


dataset, state = build_dataset(matches)
print(f"Feature rows: {len(dataset):,}")
dataset.head()

Feature rows: 20,001


,rank_diff,rank_points_diff,age_diff,height_diff,form_diff,surface_form_diff,h2h_a_winrate,label,tourney_date
0,-7.0,-1613.0,-6.2,20.0,0.0,0.0,0.5,0,2018-12-31
1,223.0,1777.0,-10.9,10.0,-0.5,-0.5,0.5,1,2018-12-31
2,-31.0,-2540.0,2.8,10.0,-0.5,-0.5,0.5,0,2018-12-31
3,208.0,1098.0,-13.9,-5.0,0.5,0.5,0.5,0,2018-12-31
4,2.0,122.0,-5.2,2.0,0.0,0.0,0.5,1,2018-12-31


## 3. Honest evaluation
Raw accuracy flatters a model, because the higher-ranked player wins about 64% of ATP matches anyway. So this section:

- tests on the **most recent** 20% of matches (the model never sees the future while training),
- compares against two baselines: "always pick the higher-ranked player" and a ranking-only logistic regression,
- scores **probabilities** with log loss and Brier score, not just picks,
- uses a **bootstrap** to check whether the improvement is real or noise.

In [5]:
def match_loss(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def bootstrap_ci(values_a, values_b, reps=2000, seed=0):
    """95% CI for mean(a) - mean(b), resampling matches (a and b are paired per match)."""
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values_a), (reps, len(values_a)))
    d = values_a[idx].mean(axis=1) - values_b[idx].mean(axis=1)
    return float(d.mean()), [float(x) for x in np.percentile(d, [2.5, 97.5])]


# Chronological split: train on the past, test on the future
split = int(len(dataset) * TRAIN_FRAC)
train, test = dataset.iloc[:split], dataset.iloc[split:].copy()
X_tr, y_tr, X_te, y_te = train[FEATURES], train["label"], test[FEATURES], test["label"]
y = y_te.values
print(f"Train: {len(train):,} matches | Test: {len(test):,} matches "
      f"({test['tourney_date'].min().date()} to {test['tourney_date'].max().date()})")

rank_cols = ["rank_diff", "rank_points_diff"]
gbm = make_model().fit(X_tr, y_tr)
rank_lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr[rank_cols], y_tr)
p_gbm = gbm.predict_proba(X_te)[:, 1]
p_lr = rank_lr.predict_proba(X_te[rank_cols])[:, 1]
p_coin = np.full(len(y), y_tr.mean())
rank_pick = (X_te["rank_diff"] > 0).astype(int).values

rows = []
for name, p, hard in [("coin flip (base rate)", p_coin, None),
                      ("always higher-ranked", None, rank_pick),
                      ("rank-only logistic", p_lr, None),
                      ("GBM (this model)", p_gbm, None)]:
    pred = hard if hard is not None else (p >= 0.5).astype(int)
    r = {"model": name, "accuracy": accuracy_score(y, pred)}
    if p is not None:
        r.update(logloss=log_loss(y, p), brier=brier_score_loss(y, p), auc=roc_auc_score(y, p))
    rows.append(r)
pd.DataFrame(rows).set_index("model").round(3)

Train: 16,000 matches | Test: 4,001 matches (2025-01-30 to 2026-05-25)


,accuracy,logloss,brier,auc
model,,,,
coin flip (base rate),0.510,0.693,0.250,0.500
always higher-ranked,0.642,NaN,NaN,NaN
rank-only logistic,0.641,0.634,0.221,0.695
GBM (this model),0.653,0.628,0.219,0.702


In [6]:
# Is the improvement over the baselines real, or noise?
gain, gain_ci = bootstrap_ci(match_loss(y, p_lr), match_loss(y, p_gbm))
gbm_ok = ((p_gbm >= .5).astype(int) == y).astype(float)
rank_ok = (rank_pick == y).astype(float)
edge, edge_ci = bootstrap_ci(gbm_ok, rank_ok)
print(f"Log loss improvement vs rank-only model: {gain:.4f}   95% CI ({gain_ci[0]:.4f}, {gain_ci[1]:.4f})")
print(f"Accuracy edge vs 'higher-ranked' rule:   {edge:.4f}   95% CI ({edge_ci[0]:.4f}, {edge_ci[1]:.4f})")
print("(If an interval excludes 0, the improvement is statistically distinguishable from zero.)")

metrics = {
    "n_test": int(len(test)),
    "test_start": str(test["tourney_date"].min().date()),
    "test_end": str(test["tourney_date"].max().date()),
    "gbm_accuracy": float(accuracy_score(y, p_gbm >= .5)),
    "gbm_logloss": float(log_loss(y, p_gbm)),
    "rank_accuracy": float(accuracy_score(y, rank_pick)),
    "rank_lr_logloss": float(log_loss(y, p_lr)),
    "logloss_gain": gain, "logloss_gain_ci": gain_ci,
    "accuracy_edge": edge, "accuracy_edge_ci": edge_ci,
}

Log loss improvement vs rank-only model: 0.0063   95% CI (0.0016, 0.0112)
Accuracy edge vs 'higher-ranked' rule:   0.0110   95% CI (0.0012, 0.0205)
(If an interval excludes 0, the improvement is statistically distinguishable from zero.)


In [7]:
# Accuracy by how lopsided the match is (does the model beat "pick the favorite" in close matches?)
test["gbm_correct"], test["rank_correct"] = gbm_ok, rank_ok
test["rank_gap"] = pd.cut(test["rank_diff"].abs(), [-1, 10, 25, 50, 100, 10_000],
                          labels=["0-10", "11-25", "26-50", "51-100", "100+"])
by_gap = test.groupby("rank_gap", observed=True).agg(
    matches=("label", "size"), gbm_acc=("gbm_correct", "mean"), rank_acc=("rank_correct", "mean"))
by_gap.round(3)

,matches,gbm_acc,rank_acc
rank_gap,,,
0-10,553,0.604,0.561
11-25,760,0.603,0.599
26-50,928,0.654,0.633
51-100,979,0.688,0.691
100+,781,0.689,0.690


In [8]:
# Calibration: when the model says X%, does it happen X% of the time?
frac, mean_pred = calibration_curve(y, p_gbm, n_bins=10, strategy="quantile")
print("Predicted -> actual win rate")
for mp, fp in zip(mean_pred, frac):
    print(f"  {mp:.2f} -> {fp:.2f}")

perm = permutation_importance(gbm, X_te, y_te, n_repeats=10, random_state=SEED, scoring="roc_auc")
print("\nFeature importance (drop in AUC when the feature is shuffled):")
for f, v in sorted(zip(FEATURES, perm.importances_mean), key=lambda t: -t[1]):
    print(f"  {f:20s} {v:.3f}")

Predicted -> actual win rate
  0.19 -> 0.22
  0.29 -> 0.29
  0.36 -> 0.36
  0.42 -> 0.37
  0.47 -> 0.46
  0.52 -> 0.53
  0.57 -> 0.55
  0.63 -> 0.64
  0.71 -> 0.71
  0.82 -> 0.78



Feature importance (drop in AUC when the feature is shuffled):
  rank_points_diff     0.043
  rank_diff            0.040
  surface_form_diff    0.009
  age_diff             0.005
  h2h_a_winrate        0.002
  height_diff          0.001
  form_diff            0.000


## 4. Final model and predictions
The final model trains on **all** matches. Predictions use each player's most recent rank, points, age, height and match history. The model can answer slightly differently depending on which player is "A", so `predict_match` averages both orderings; the two probabilities always add up to 100%.

In [9]:
model = make_model().fit(dataset[FEATURES], dataset["label"])
snapshots = latest_snapshots(matches)


def _player(name):
    if name not in snapshots.index:
        raise ValueError(f"No matches found for player: {name!r}")
    s = snapshots.loc[name]
    return Player(s["id"], s["rank"], s["pts"], s["age"], s["ht"])


def win_probabilities(pairs, surface="Hard", symmetric=True):
    """P(first player beats second) for each (name_a, name_b) pair."""
    def proba(ps):
        rows = [pair_features(state, _player(a), _player(b), surface) for a, b in ps]
        return model.predict_proba(pd.DataFrame(rows)[FEATURES])[:, 1]
    forward = proba(pairs)
    if not symmetric:
        return forward
    return (forward + 1 - proba([(b, a) for a, b in pairs])) / 2


def predict_match(player_a, player_b, surface="Hard"):
    p = float(win_probabilities([(player_a, player_b)], surface)[0])
    return {"player_a": player_a, "player_b": player_b, "surface": surface,
            "prob_player_a_wins": round(p, 3), "prob_player_b_wins": round(1 - p, 3),
            "favorite": player_a if p >= 0.5 else player_b}


print(f"Model trained on {len(dataset):,} matches through {matches['tourney_date'].max().date()}\n")
for a, b, s in [("Jannik Sinner", "Carlos Alcaraz", "Clay"), ("Novak Djokovic", "Daniil Medvedev", "Grass")]:
    try:
        print(predict_match(a, b, s))
    except ValueError as e:
        print(e)

Model trained on 20,001 matches through 2026-05-25

{'player_a': 'Jannik Sinner', 'player_b': 'Carlos Alcaraz', 'surface': 'Clay', 'prob_player_a_wins': 0.562, 'prob_player_b_wins': 0.438, 'favorite': 'Jannik Sinner'}
{'player_a': 'Novak Djokovic', 'player_b': 'Daniil Medvedev', 'surface': 'Grass', 'prob_player_a_wins': 0.57, 'prob_player_b_wins': 0.43, 'favorite': 'Novak Djokovic'}


### Try your own matchup
Change the names and surface (`Hard`, `Clay`, `Grass`, or `Carpet`) and run this cell. Names must match the data's spelling, for example `Jannik Sinner`.

In [10]:
PLAYER_A = "Jannik Sinner"
PLAYER_B = "Carlos Alcaraz"
SURFACE = "Hard"

try:
    r = predict_match(PLAYER_A, PLAYER_B, SURFACE)
    print(f"{r['player_a']}: {r['prob_player_a_wins']:.1%}   {r['player_b']}: {r['prob_player_b_wins']:.1%}   ({SURFACE})")
except ValueError as e:
    print(e)
    print("Tip: search names with, e.g.  [n for n in snapshots.index if 'Sinner' in n]")

Jannik Sinner: 59.6%   Carlos Alcaraz: 40.4%   (Hard)


## 5. Sanity checks
Quick tests that the pipeline behaves correctly. If any fails, the notebook stops with an error.

In [11]:
# 1) The better-ranked player is favored, and probabilities are symmetric and valid
top = snapshots.sort_values("rank").index[:30]
pairs = [(top[0], top[20]), (top[5], top[25]), (top[10], top[2])]
fwd = win_probabilities(pairs, "Hard")
rev = win_probabilities([(b, a) for a, b in pairs], "Hard")
assert np.allclose(fwd + rev, 1.0), "A-vs-B and B-vs-A should sum to 1"
assert ((fwd > 0) & (fwd < 1)).all(), "probabilities must be strictly between 0 and 1"
assert fwd[0] > 0.5 and fwd[1] > 0.5 and fwd[2] < 0.5, "better-ranked player should be favored"

# 2) No leakage: a match's features must not change when LATER matches are added
k = 300
prefix, _ = build_dataset(matches.iloc[: k + 1])
pd.testing.assert_series_equal(prefix.iloc[k][FEATURES], dataset.iloc[k][FEATURES], check_names=False)

# 3) Random side assignment: labels should be roughly balanced
assert 0.45 < dataset["label"].mean() < 0.55, "labels should be ~50/50"

# 4) Every match recorded exactly one win/loss result per player
assert sum(len(v) for v in state.results.values()) == 2 * len(matches)

# 5) First match has no history
first = dataset.iloc[0]
assert first["form_diff"] == 0 and first["h2h_a_winrate"] == 0.5

# 6) Unknown players raise a clear error
try:
    predict_match("Nobody Real", top[0])
    raise AssertionError("expected ValueError")
except ValueError:
    pass

print("All sanity checks passed.")

All sanity checks passed.


## 6. Website export (legacy, optional)
The website no longer uses this. It now loads a trained model from `web/public/model.json`, which `python/export_model.py` builds (and the GitHub Actions workflow runs on every deploy). The cell below is the older approach: a precomputed win-probability table for the top 80 players, saved as `predictions.json`. Skip it unless you want that table for something else.

In [12]:
TOP = 80
SURFACES = ["Hard", "Clay", "Grass"]
asof = matches["tourney_date"].max()
active = (snapshots[snapshots["date"] >= asof - pd.Timedelta(days=270)]
          .sort_values("rank").head(TOP))
names = list(active.index)
n = len(names)

# Pairs in (i < j) order; the website reads them back with the same index formula
pairs = [(names[i], names[j]) for i in range(n) for j in range(i + 1, n)]
probs = {s: [int(round(1000 * p)) for p in win_probabilities(pairs, s)] for s in SURFACES}

site_data = {
    "asof": str(asof.date()),
    "players": [{"name": nm, "rank": int(r["rank"]), "points": int(r["pts"]),
                 "age": round(float(r["age"]), 1)} for nm, r in active.iterrows()],
    "probs": probs,
    "metrics": metrics,
}
with open("predictions.json", "w") as f:
    json.dump(site_data, f, separators=(",", ":"))
print(f"Saved predictions.json: {n} players, {len(pairs):,} pairs per surface")

try:                                   # in Colab, also download it
    from google.colab import files
    files.download("predictions.json")
except ImportError:
    print("Saved in the notebook's current folder.")

Saved predictions.json: 80 players, 3,160 pairs per surface
Saved in the notebook's current folder.
